##Chapter 5:
* Feature Engineering on Databricks/Project: Favorita Store Sales - Time Series Forecasting/CH5-01-Building Favorita Feature Tables.py

In [0]:
%pip install -q databricks-feature-engineering

In [0]:
from databricks.feature_engineering.client import FeatureEngineeringClient

fe = FeatureEngineeringClient()

In [0]:
sql("""
  CREATE OR REPLACE TABLE porya_catalog.default.local_holidays_bronze AS (
    SELECT
      `date`,
      `type` as holiday_type,
      locale_name
    FROM
      porya_catalog.default.holiday_events
    WHERE
      locale == "Local"
    GROUP BY
      ALL
    ORDER BY
      `date`)
""")

In [0]:

sql(
"""
CREATE OR REPLACE TABLE porya_catalog.default.local_holidays_silver AS (
SELECT
  `date`, store_nbr,
  CASE
    WHEN num_holidays >= 2 THEN "Multiple"
    ELSE local_holiday_type
  END as local_holiday_type
FROM
  (
    SELECT
      h.`date`, s.store_nbr,
      MIN(h.holiday_type) as local_holiday_type,
      count(1) as num_holidays
    FROM
      porya_catalog.default.favorita_stores s
      INNER JOIN porya_catalog.default.local_holidays_bronze h ON (
        s.city = h.locale_name
        OR s.state = h.locale_name
      )
    GROUP BY
      ALL
    ORDER BY
      h.`date`
  )
)
""")
display(sql("SELECT * FROM porya_catalog.default.local_holidays_silver LIMIT 5"))

# COMMAND ----------

# DBTITLE 1,Regional holidays
sql("""
  CREATE OR REPLACE TABLE porya_catalog.default.regional_holidays_bronze AS (
    SELECT
      `date`,
      `type` as holiday_type,
      locale_name
    FROM
      porya_catalog.default.holiday_events
    WHERE
      locale = "Regional"
    GROUP BY
      ALL
    ORDER BY
      `date`)
""")

sql(
"""
CREATE OR REPLACE TABLE porya_catalog.default.regional_holidays_silver AS (
SELECT
  `date`,
  store_nbr,
  CASE
    WHEN num_holidays >= 2 THEN "Multiple"
    ELSE regional_holiday_type
  END as regional_holiday_type
FROM
  (
    SELECT
      h.`date`,
      s.store_nbr,
      MIN(h.holiday_type) as regional_holiday_type,
      count(1) as num_holidays
    FROM
      porya_catalog.default.favorita_stores s
      INNER JOIN porya_catalog.default.regional_holidays_bronze h ON (
        s.city = h.locale_name
        OR s.state = h.locale_name
      )
    GROUP BY
      ALL
    ORDER BY
      h.`date`
  )
)
""")
display(sql("SELECT * FROM porya_catalog.default.regional_holidays_silver LIMIT 5"))

# COMMAND ----------

# DBTITLE 1,National holidays
sql("""
  CREATE OR REPLACE TABLE porya_catalog.default.national_holidays_bronze AS (
    SELECT
      `date`,
      MIN(`type`) as holiday_type,
      count(1) as num_holidays
    FROM
      porya_catalog.default.holiday_events
    WHERE
      locale = "National"
    GROUP BY ALL
    ORDER BY
      `date`)
""")

sql(
"""
CREATE OR REPLACE TABLE porya_catalog.default.national_holidays_silver AS (
SELECT
  `date`,
  store_nbr,
  national_holiday_type
FROM
  (
    SELECT
      `date`,
      CASE
        WHEN num_holidays >= 2 THEN "Multiple"
        ELSE holiday_type
      END as national_holiday_type
    FROM
      porya_catalog.default.national_holidays_bronze h
    GROUP BY
      ALL
    ORDER BY
      h.`date`
  ), porya_catalog.default.favorita_stores 
)
""")
display(sql("SELECT * FROM porya_catalog.default.national_holidays_silver LIMIT 5"))

# COMMAND ----------

# DBTITLE 1,Holiday feature table
df = sql("""
      SELECT
        ifnull(n.`date`,ifnull(r.`date`, l.`date`)) as `date`,
        ifnull(n.store_nbr,ifnull(r.store_nbr, l.store_nbr)) as store_nbr,
        n.national_holiday_type,
        r.regional_holiday_type,
        l.local_holiday_type
      FROM
        porya_catalog.default.national_holidays_silver n
      FULL JOIN porya_catalog.default.regional_holidays_silver r ON n.`date`=r.`date` AND n.store_nbr = r.store_nbr
      FULL JOIN porya_catalog.default.local_holidays_silver l ON n.`date`=l.`date` AND n.store_nbr = l.store_nbr
      ORDER BY
        `date`
      """)

df.limit(5).display()

fe.create_table(
    name = f"porya_catalog.default.store_holidays_ft",
    primary_keys=["date", "store_nbr"],
    df=df,
    description="Holidays in Ecuador by date and store number. Table includes holiday types for national, regional, and local. Days where a store has more than one holiday is indicated by holiday type being 'Multiple'. Nulls indicate non-holiday days.",
)

# COMMAND ----------

# MAGIC %sql
# MAGIC
# MAGIC SELECT * FROM porya_catalog.default.store_holidays_ft

# COMMAND ----------

# DBTITLE 1,Lagging the oil price for a proxy of the economy
df = sql(
    """
    SELECT
      timestamp(`date`) as price_date,
      timestamp(date(`date`) -10) as date,
      dcoilwtico as lag10_oil_price
    FROM
      porya_catalog.default.oil_prices_silver
  """
)

fe.create_table(
    name=f"porya_catalog.default.oil_10d_lag_ft",
    primary_keys=["date"],
    df=df,
    description="The lag10_oil_price is the price of oil 10 days after date. The price date is the actual date of the oil price. The oil prices were imputed to replace nulls with the previous day's price. The stock market is not open on weekends and holidays.",
)